# DESC Sprint Week ELAsTiCC Tutorial Demo 2

## Reading ELAsTiCC2 Parquet files

This is probably the best general-use format, if you don't already have code that processes SNANA FITS files.

The structure of the Parquet files means that they will tend to work better with Polars dataframes, but you can also use them with Pandas dataframes.  (Both provide a `read_parquet` function.)

On NERSC, the parquet files can be found at `/global/cfs/cdirs/desc-td/ELASTICC2_parquet`

In [ ]:
%matplotlib inline

import sys
import math
import os
import pathlib
import logging

import numpy
import polars
import matplotlib
from matplotlib import pyplot

# Need to add the lib_elasticc2 directory to the PYTHONPATH so that we
#  can import it
libdir = pathlib.Path( os.getcwd() ).parent.parent / "lib_elasticc2"
sys.path.insert( 0, str(libdir) )
from read_snana import elasticc2_snana_reader

# Make a logger so that we can print out timings and things like that
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler( sys.stderr )
    _logger.addHandler( _logout )
    _logout.setFormatter( logging.Formatter( f'[%(asctime)s - %(levelname)s] - %(message)s',
                                             datefmt='%Y-%m-%d %H:%M:%S' ) )
_logger.setLevel( logging.INFO )
_logger.info( "Testing" )

# Make a random number generator.  If you 
#  want reproducibility, set the seed
#  to something other than None.
# _random_seed = None
_random_seed = 42
rng = numpy.random.default_rng( seed=_random_seed )

PHOTFLAG_DETECT = 0x1000
PHOTFLAG_TRIGGER = 0x0800
PHOTFLAG_SATURATE = 0x0400

In [ ]:
# Define a function for plotting lightcurves.  We're going to use
# this lots below, and this saves repeated code in cells
#
# This version of plot_ltcv expects Polars serieses for mjd, band, flux, and fluxerr
def plot_ltcv( mjd, band, flux, fluxerr, snid=None, zcmb=None, mjdoff=0, figsize=None, width=None, multiplots=False ):
    plotcolors = { 'u': '#cc0ccc', 
                   'g': '#00cc44', 
                   'r': '#cc0000', 
                   'i': '#ff4400', 
                   'z': '#886600',
                   'Y': '#442200' }
    knownbands = band.unique()
    if any( b not in plotcolors.keys() for b in knownbands ):
        _logger.warning( f"Unknown bands not plotted: {[b for b in knownbands if b not in plotcolors.keys()]}" )
    bandstoplot = [ b for b in plotcolors.keys() if b in knownbands ]
    
    if multiplots:
        nrows = math.ceil( len(bandstoplot) / 2 )
        if figsize is None:
            width = 9 if width is None else width
            figsize = ( width, width/3. * nrows )
        fig, axes = pyplot.subplots( nrows, 2, figsize=figsize, tight_layout=True, sharex='all' )
        axes = axes.flatten()
    else:
        if figsize is None:
            width = 9 if width is None else width
            figsize = ( width, width/2. )
        fig, axes = pyplot.subplots( 1, 1, figsize=figsize, tight_layout=True )
        axes = [ axes ]
    axesdex = 0
    
    for curband in bandstoplot:
        inband = ( band == curband )
        axes[axesdex].errorbar( mjd.filter(inband)-mjdoff, flux.filter(inband), yerr=fluxerr.filter(inband),
                                color=plotcolors[curband], linestyle='None', marker='o',
                                label=curband )
        if multiplots: axesdex += 1

    for i, axis in enumerate(axes):
        if i >= len(bandstoplot):
            axis.set_visible( False )
        else:
            title = ""
            if snid is not None: title += f"SN {snid}"
            if zcmb is not None: title += f"{' at ' if snid is not None else ''} z={zcmb:.3f}"
            if len(title) > 0: axis.set_title( title )
            if mjdoff != 0:
                axis.set_xlabel( f"MJD-{mjdoff}" )
            else:
                axis.set_xlabel( r"MJD" )
            axis.set_ylabel( r"Flux" )
            axis.tick_params( axis='both', reset=True )
            axis.legend()

    # In the jupyter script environment, the plot gets shown
    #   inline automatically.  If you're running this
    #   from the command line, you might need to do
    #   fig.show().  You might also want to do something
    #   like fig.savefig(filename).  So, return the Figure
    #   to make these things possible.
    # One side-effect of this is that your figure may be
    #   shown *twice* in your jupyter notebook; once for
    #   the plotting above, and once again if the call
    #   to plot_ltcv is the last command in the cell,
    #   because jupyter by default displays the value
    #   of the last expression in each cell.  Add a ;
    #   to the end of your plot_ltcv cell to supporess
    #   this in jupyter.
    return fig


In [ ]:
# Read one of the parquet files into a polars data frame.
# For a large model, this will take a long time (~0.5-3 minutes),
# though not nearly as long as it would take to read all of
# the gzipped SNANA FITS files for a single large model.

pqfile = ( pathlib.Path(os.getenv("TD", "/global/cfs/cdirs/lsst/groups/TD"))
           / "ELASTICC2_parquet/SNIa-SALT3.parquet" )
_logger.info( f"Reading {pqfile}..." )
sniadf = polars.read_parquet( pqfile )
_logger.info( f"...done reading {pqfile.name}; the polars dataframe uses ~{sniadf.estimated_size('gb'):.3f} GB of memory." )

In [ ]:
# The structure of one of these Polars data frames is different from
# that of a Pandas data frame (or even a Polars data frame) read with
# the elasticc2_snana_reader library.  There are two primary differences.
# First, there are many columns, including a lot of the object truth
# data.  (The object truth data is mostly in the "SIM_*" columns, such
# as SIM_REDSHIFT_CMB.)  Second, lightcurves are included as list columns.
# Pandas doesn't really like you to use this kind of structure; Polars
# has more built-in support for columns whose elements are lists.

sniadf[0:5]

In [ ]:
print( f"The first object in the list is SNID {sniadf['SNID'][0]}" )
print( f"It has a CMB-z of {sniadf['SIM_REDSHIFT_CMB'][0]}" )
mjd = sniadf['MJD'][0]
band = sniadf['BAND'][0]
fluxcal = sniadf['FLUXCAL'][0]
fluxcalerr = sniadf['FLUXCALERR'][0]
photflag = sniadf['PHOTFLAG'][0]

# Rather than just showing the detections, let's show all the forced photometry
# from 20 days before first detection to 20 days after last detection
detects = ( photflag & PHOTFLAG_DETECT ) != 0
mjdmin = mjd.filter( detects ).min() - 20
mjdmax = mjd.filter( detects ).max() + 20
toplot = ( mjd >= mjdmin ) & ( mjd <= mjdmax )

plot_ltcv( mjd.filter(toplot), band.filter(toplot), fluxcal.filter(toplot), fluxcalerr.filter(toplot),
           snid=sniadf['SNID'][0], zcmb=sniadf['SIM_REDSHIFT_CMB'][0],
           mjdoff=sniadf['PEAKMJD'][0], multiplots=True, width=8 );

In [ ]:
# Histogram of redshifts (from the truth table)
fig, axes = pyplot.subplots( 1, 1, tight_layout=True )
axes.hist( sniadf['SIM_REDSHIFT_CMB'], bins=numpy.arange( 0, 1.7, 0.1 ) )
axes.set_xlabel( r"z_CMB" )
axes.set_ylabel( r"N" )
_;